## this script cleans the gender information of authors. for each cluster_id, choose the most frequent gender category.

### input
- author_info: the data is derived from [BDVincent].[dbo].[gender_prod_80]

### output
- pub_author: the file maps paper id with author id
- author_info: the file contains demographic information of authors

In [1]:
import pandas as pd
from dask import dataframe as dd
import collections

In [2]:
info_path='../../data/dropbox/LaborDivision/Data/rawdata/author_info.txt'

In [43]:
pub_author_path='../../data/dropbox/LaborDivision/Data/cleanedraw/pub_authorid.pkl'
info_new_path='../../data/dropbox/LaborDivision/Data/cleanedraw/author_info.pkl'

In [30]:
info_df= pd.read_csv(info_path,sep="\t")
info_df=info_df.rename(columns={'OST_BK':'id','Seq_No':'seqno','YFP':'yfp','npap':'npub'})
info_df.head()

,id,seqno,first_name,cluster_id,gender,yfp,npub
0,75871588,2,S. H.,689940,INI,2008,1
1,53614014,3,U.,44823147,INI,2008,1
2,78030267,1,Rui,27676457,M,2011,1
3,54383423,2,Madeleine,59751143,F,2018,1
4,32237142,3,Frank,59894652,M,2004,8


## extract the paper_author mapping relation

In [14]:
pub_author=info_df[['id','seqno','cluster_id']]
pub_author.to_pickle(pub_author_path)

## assign authors to its most frequent gender

In [31]:
gender_df=info_df[['cluster_id','first_name','gender']].drop_duplicates()

In [32]:
# standardize
gender_df=gender_df.replace(to_replace={'gender':{'m':'M','f':'F','INI':'unk','UNI':'unk','UNK':'unk'}})
gender_df=gender_df.fillna(value='unk')

In [33]:
gender_df = gender_df.groupby(['cluster_id','gender']).size().reset_index(name='count')
gender_df=gender_df.pivot_table(index='cluster_id',columns='gender',values='count')
gender_df=gender_df.fillna(0)
id_gender=gender_df.idxmax(axis=1).reset_index(name='gender')#return the column has largest value

In [21]:
info_df.head()

,id,seqno,first_name,cluster_id,gender,yfp,npub
0,75871588,2,S. H.,689940,INI,2008,1
1,53614014,3,U.,44823147,INI,2008,1
2,78030267,1,Rui,27676457,M,2011,1
3,54383423,2,Madeleine,59751143,F,2018,1
4,32237142,3,Frank,59894652,M,2004,8


In [34]:
id_gender.gender.unique()

array(['M', 'F', 'unk'], dtype=object)

In [35]:
info_new=info_df[['cluster_id','yfp','npub']]
info_new=info_new.drop_duplicates(subset=['cluster_id'])

In [36]:
info_new=info_new.merge(id_gender,on='cluster_id')

In [37]:
## count the gender identification distribution of authors
collections.Counter(info_new['gender'].values.tolist())

Counter({'unk': 5610284, 'M': 12550338, 'F': 7275226})

In [38]:
5610284/(5610284+12550338+7275226)

0.22056602948720247

In [40]:
info_new.head()

,cluster_id,yfp,npub,gender
0,689940,2008,1,unk
1,44823147,2008,1,unk
2,27676457,2011,1,M
3,59751143,2018,1,F
4,59894652,2004,8,M


In [41]:
info_new=info_new.astype({'yfp':'int16','npub':'int16','cluster_id':'str'})

In [44]:
info_new.to_pickle(info_new_path)